# Módulo 3 · DataFrames en PySpark

⏱ **16 min** · Slides: 20–23 (particiones, DataFrames, transformaciones y acciones, esquema)

**Objetivo:** comprobar con experimentos las tres propiedades de un DataFrame: **distribuido, inmutable y lazy**.

| Paso | Concepto | Evidencia |
|---|---|---|
| 1 | Esquema: inferir vs explícito | Tiempos y tipos |
| 2 | Distribuido | Filas por partición |
| 3 | Inmutable | El original no cambia |
| 4 | Lazy | Definir ≈ 0 s, ejecutar > 0 s |
| 5 | Transformación vs acción | Qué devuelve cada una |
| 6 | Plan de ejecución y shuffle | `explain()` |
| 7 | El caso `007 → 7` | Inferencia que se equivoca |
| 8 | Tu turno | Consulta propia |

**Requiere:** módulo 2 (archivos en RAW).

In [0]:
%run ./00_Setup

## Paso 1 · Esquema: inferir vs explícito

Regla práctica: **inferir para explorar; esquema explícito para pipelines.**

> **Ojo al medir:** `spark.read` es *lazy*, así que por sí solo tarda ~0 ms en ambos casos.
> El costo de inferir aparece cuando Spark necesita **conocer el esquema** (`df.schema`) y cuando ejecutas una acción (`count()`).
> Por eso medimos esos dos momentos, en milisegundos. Ejecuta las dos celdas **dos veces**: la primera puede incluir el arranque del cómputo.

In [0]:
# Opción A: inferir. Para adivinar los tipos, Spark tiene que leer los archivos
t0 = time.perf_counter()
df_infer = spark.read.option("multiLine", True).json(RAW_GLOB)
_ = df_infer.schema                                  # aquí Spark infiere los tipos
ms_infer_esquema = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
df_infer.count()                                     # acción
ms_infer_count = (time.perf_counter() - t0) * 1000

print(f"Con inferencia → resolver esquema: {ms_infer_esquema:,.0f} ms | count(): {ms_infer_count:,.0f} ms")
df_infer.select("id", "name", "number", "runtime", "rating").printSchema()

In [0]:
# Opción B: esquema explícito (definido en 00_Setup). No necesita adivinar nada
t0 = time.perf_counter()
df = (spark.read.format("json")
      .option("multiLine", True)
      .schema(ESQUEMA_EPISODIOS)
      .load(RAW_GLOB))
_ = df.schema                                        # ya lo conoce: no lee nada
ms_exp_esquema = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
df.count()                                           # acción
ms_exp_count = (time.perf_counter() - t0) * 1000

print(f"Explícito      → resolver esquema: {ms_exp_esquema:,.0f} ms | count(): {ms_exp_count:,.0f} ms")
df.select("id", "name", "number", "runtime", "rating").printSchema()

👀 **Qué observar:**
- **Resolver el esquema:** con inferencia es mucho mayor (tuvo que leer los archivos); con esquema explícito es casi instantáneo.
- **`count()`:** con inferencia ya pagaste la lectura extra; con esquema explícito solo se lee una vez.
- **Tipos:** los inferidos pueden diferir de los que tú quieres (por ejemplo `long` donde necesitas `int`). Con esquema explícito son predecibles.

Si la diferencia es pequeña, es porque hay pocos datos: sube el widget `dias` del módulo 2 a `30` y vuelve a medir.

De aquí en adelante usamos `df` (el explícito).

## Paso 2 · Distribuido: los datos viven en particiones

In [0]:
por_particion = (df.withColumn("particion", F.spark_partition_id())
                   .groupBy("particion").count()
                   .orderBy("particion"))
display(por_particion)

👀 **Qué observar:** hay pocas particiones porque los datos son pequeños (Spark crea ~1 por cada 128 MB).
Con un archivo de 10 GB serían unas 80 particiones repartidas entre los workers. Forcemos un reparto:

In [0]:
display(df.repartition(8)
          .withColumn("particion", F.spark_partition_id())
          .groupBy("particion").count()
          .orderBy("particion"))

## Paso 3 · Inmutable: cada transformación crea un DataFrame nuevo

In [0]:
df_largo = df.withColumn("es_largo", F.col("runtime") > 45)

print("Columnas de df       :", len(df.columns))
print("Columnas de df_largo :", len(df_largo.columns))
print("¿df tiene 'es_largo'?:", "es_largo" in df.columns)

👀 **Qué observar:** `df` quedó intacto. `withColumn` no modifica: devuelve otro DataFrame.

## Paso 4 · Lazy: nada corre hasta que hay una acción

In [0]:
t0 = time.time()
plan = (df.filter(F.col("runtime") > 30)
          .groupBy("airdate").count()
          .orderBy("airdate"))
print(f"Definir las transformaciones: {time.time() - t0:.2f} s   (no se ejecutó nada)")

In [0]:
t0 = time.time()
plan.show()
print(f"Ejecutar con show():          {time.time() - t0:.2f} s   (aquí sí trabajó el cómputo)")

## Paso 5 · Transformación vs acción

| Transformaciones (lazy) | Acciones (disparan la ejecución) |
|---|---|
| `select`, `filter`, `withColumn`, `groupBy`, `join` | `display`, `count`, `show`, `write` |

In [0]:
una_transformacion = df.filter(F.col("runtime") > 30)
una_accion         = df.count()

print("filter() devolvió:", type(una_transformacion).__name__)
print("count()  devolvió:", type(una_accion).__name__, "→", una_accion, "filas")

👀 **Qué observar:** `filter()` devuelve un DataFrame (solo un plan); `count()` devuelve un número real.
Por eso `display()` "ejecuta algo" y `filter()` no (pregunta 3 del quiz).

## Paso 6 · El plan de ejecución y el shuffle

Antes de ejecutar, el optimizador **Catalyst** arma un plan. `explain()` lo muestra sin ejecutarlo.

In [0]:
# Sin groupBy: no hay movimiento de datos entre workers
df.filter(F.col("runtime") > 30).select("id", "name").explain()

In [0]:
# Con groupBy: Spark debe reunir todas las filas de cada clave → shuffle
df.groupBy("airdate").count().explain()

👀 **Qué observar:** en el segundo plan aparece **`Exchange`**. Esa es la palabra que marca un shuffle:
datos viajando entre workers. `groupBy` y `join` son las operaciones más costosas.

## Paso 7 · Cuando inferir se equivoca: `007 → 7`

Un CSV mínimo con códigos que empiezan en cero.

In [0]:
ruta_csv = f"{TMP_PATH}/canales.csv"
_ = dbutils.fs.put(ruta_csv, "codigo_canal,canal\n007,Seasonvar\n042,Night Channel\n", True)

inferido = (spark.read.option("header", True).option("inferSchema", True).csv(ruta_csv))
explicito = (spark.read.option("header", True)
             .schema("codigo_canal STRING, canal STRING").csv(ruta_csv))

print("Con inferSchema:")
display(inferido)

In [0]:
print("Con esquema explícito:")
display(explicito)

👀 **Qué observar:** con inferencia `007` se convirtió en `7` (se perdió información). Con esquema explícito se conserva.

## Paso 8 · Tu turno

Con `df`, obtén las **10 series con más episodios** en el periodo descargado.

*Pista:* el nombre de la serie está en `_embedded.show.name`. Necesitas `groupBy` → `count` → `orderBy` → `limit`.

In [0]:
# Escribe tu solución aquí

### Solución

In [0]:
top10 = (df.groupBy(F.col("_embedded.show.name").alias("serie"))
           .count()
           .orderBy(F.col("count").desc())
           .limit(10))
display(top10)

## ✔ Checkpoint

- [ ] Distribuido: veo las filas repartidas en particiones
- [ ] Inmutable: `withColumn` no cambió `df`
- [ ] Lazy: definir tomó ≈ 0 s; ejecutar, no
- [ ] Sé encontrar un shuffle (`Exchange`) en `explain()`
- [ ] Uso esquema explícito en pipelines

**Siguiente:** `04_Formatos_CSV_Parquet_Delta`